# Notebook 2 — Extract Keypoints từ fall/sequences
**Kaggle T4x2 · ~40 phút**

In [ ]:
!pip install ultralytics -q

import os, csv, json, shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Đường dẫn gốc ─────────────────────────────────────────────────────────────
BASE    = Path('/kaggle/input/datasets/nguynteincong/dataset-fall-fire/datasets')
SEQ_DIR = BASE / 'Fall' / 'sequences'
CSV_DIR = BASE / 'Fall' / 'csv'
OUT_DIR = Path('/kaggle/working/keypoints')
OUT_DIR.mkdir(exist_ok=True)

# Kiểm tra
sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])
print(f'Số sequences: {len(sequences)}')
for s in sequences:
    n_frames = len(list(s.glob('*.png')))
    csv_name = '-'.join(s.name.split('-')[:2]) + '-data.csv'
    has_csv  = (CSV_DIR / csv_name).exists()
    print(f'  {s.name:<30} frames={n_frames:4d}  csv={has_csv}')

In [ ]:
IMG_EXTS = {'.png', '.jpg', '.jpeg'}

def read_csv(csv_path):
    mapping = {}
    with open(csv_path) as f:
        for row in csv.reader(f):
            if len(row) < 3:
                continue
            try:
                mapping[int(row[0])] = float(row[2])
            except ValueError:
                continue
    return mapping

def make_frame_labels(acc_map, n_frames, thr_fall=1.5, window=20):
    labels     = np.zeros(n_frames, dtype=np.int64)
    peak_frame = max(acc_map, key=acc_map.get, default=None)
    if peak_frame is None or acc_map[peak_frame] < thr_fall:
        return labels
    start = max(0, peak_frame - window - 1)
    end   = min(n_frames, peak_frame)
    labels[start:end] = 1
    return labels

def extract_seq_keypoints(pose_model, seq_folder):
    frames = sorted(
        [f for f in seq_folder.iterdir() if f.suffix.lower() in IMG_EXTS],
        key=lambda x: x.name
    )
    all_kps = []
    for frame in frames:
        results = pose_model(str(frame), imgsz=416, conf=0.3, verbose=False)
        r = results[0]
        if r.keypoints is not None and len(r.keypoints.xyn) > 0:
            boxes = r.boxes.xyxy.cpu().numpy()
            areas = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
            best  = int(np.argmax(areas))
            kp    = r.keypoints.xyn[best].cpu().numpy()
        else:
            kp = np.zeros((17, 2), dtype=np.float32)
        all_kps.append(kp.flatten())
    return np.array(all_kps, dtype=np.float32), frames

print('Hàm định nghĩa xong.')

In [ ]:
pose_model = YOLO('yolov8n-pose.pt')
all_seq_names = []

for seq_folder in sequences:
    csv_name = '-'.join(seq_folder.name.split('-')[:2]) + '-data.csv'
    csv_path = CSV_DIR / csv_name

    print(f'Processing {seq_folder.name}...', end=' ', flush=True)

    kps, frames = extract_seq_keypoints(pose_model, seq_folder)
    n_frames    = len(frames)

    if csv_path.exists():
        acc_map = read_csv(csv_path)
        labels  = make_frame_labels(acc_map, n_frames)
    else:
        labels = np.zeros(n_frames, dtype=np.int64)

    np.save(OUT_DIR / f'{seq_folder.name}_kps.npy',    kps)
    np.save(OUT_DIR / f'{seq_folder.name}_labels.npy', labels)

    print(f'frames={n_frames}  fall%={labels.mean()*100:.0f}%  ✓')
    all_seq_names.append(seq_folder.name)

print(f'\nHoàn thành {len(all_seq_names)} sequences.')

In [ ]:
train_seqs, val_seqs = train_test_split(all_seq_names, test_size=0.2, random_state=42)

with open(OUT_DIR / 'split.json', 'w') as f:
    json.dump({'train': train_seqs, 'val': val_seqs}, f, indent=2)

print(f'Train ({len(train_seqs)}): {train_seqs}')
print(f'Val   ({len(val_seqs)}):   {val_seqs}')

shutil.make_archive('/kaggle/working/keypoints', 'zip', '/kaggle/working/keypoints')
size_mb = os.path.getsize('/kaggle/working/keypoints.zip') / 1024 / 1024
print(f'\nkeypoints.zip: {size_mb:.1f} MB  → download rồi upload lên Kaggle Dataset')